# 🤖 AI агентыг өөрөө бүтээх — алхам алхмаар

**FitCoach (Капстоун 5-р баг)** · AI хичээлийн Colab шиг, 30–40 минут

Энэ notebook-д фитнесс дасгалжуулагч агентыг **framework-гүйгээр**, цэвэр Python-оор 5 алхамд бүтээнэ.
Яг ижил санааг FitCoach апп ашигладаг (`app/agent/coach.py`, `tools.py`) — зөвхөн "тархи" нь энд **Gemini**, апп дээр Claude.

| Алхам | Юу сурах вэ |
|---|---|
| 1 | LLM-ийг энгийнээр дуудах (чатбот) |
| 2 | **Хэрэгсэл (tool)** = Python функц |
| 3 | Модель өөрөө хэрэгсэл **сонгох** (function calling) |
| 4 | **Agent loop** — бодох → үйлдэх → үр дүнг харах → дахин бодох |
| 5 | **Санах ой** — дараагийн ярианд ч мэдэж байх |

> ▶ Нүд бүрийг дээрээс доош **Shift + Enter**-ээр ажиллуулна.

## Чатбот ба агентын ялгаа

| | Чатбот | Агент |
|---|---|---|
| Юу хийдэг | Зөвхөн текст хариулна | Хэрэгсэл дуудаж **бодит өөрчлөлт** хийнэ (өгөгдлийн сан, төлөвлөгөө) |
| Алхам | 1 асуулт → 1 хариу | Олон алхам: хэрэгслийн үр дүнг хараад дахин шийднэ |
| Санах ой | Яриа дуусмагц мартана | Чухал зүйлийг хадгалж, дараа ашиглана |

```
Хэрэглэгч ──► LLM ──► хэрэгсэл хэрэгтэй юу? ──тийм──► Python функц ажиллана
                ▲                                         │
                └──────────── үр дүнг буцааж өгнө ◄────────┘
                (хэрэгсэл хэрэггүй болмогц эцсийн хариу)
```

## 0. Бэлтгэл

1. https://aistudio.google.com → **Get API key** → **Create API key** (сургуулийн @nsu.ac.kr аккаунт байж болно)
2. Colab-ын зүүн талын **🔑 (Secrets)** → **Add new secret** → нэр: `GEMINI_API_KEY`, утга: түлхүүр → **Notebook access**-ийг асаана

Түлхүүр байхгүй бол notebook **демо горим**-оор ажиллана (энгийн дүрмээр хариулна) — FitCoach-ийн демо горимтой адил.
Түлхүүрийг кодонд хэзээ ч шууд бүү бич.

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.genai import types

MODEL = "gemini-3.5-flash-lite"   # алдаа гарвал "gemini-3.8-flash" болгож үзээрэй

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = None

client = genai.Client(api_key=API_KEY) if API_KEY else None
print("✅ Жинхэнэ Gemini" if client else "🟡 Демо горим (түлхүүргүй)")

## Алхам 1. LLM-ийг энгийнээр дуудах — энэ бол **чатбот**

Модель зөвхөн текст буцаана. Таны дасгалын төлөвлөгөөг мэдэхгүй, юу ч өөрчилж чадахгүй.

In [ ]:
question = "Миний даваа гарагийн дасгал юу вэ?"
if client:
    r = client.models.generate_content(model=MODEL, contents=question)
    print(r.text)
else:
    print("(демо) Уучлаарай, би таны төлөвлөгөөг мэдэхгүй байна.")
# 👉 Модель таны өгөгдлийг хараагүй тул ерөнхий, таамаг хариулт өгнө. Үүнийг хэрэгслээр засна.

## Алхам 2. Хэрэгсэл = энгийн Python функц

Жинхэнэ апп-д өгөгдлийн сан (Neon Postgres) байдаг. Энд түүнийг энгийн `dict`-ээр орлуулна.
**Docstring ба type hint чухал** — модель тэдгээрийг уншаад хэрэгслийг хэзээ, яаж дуудахаа шийднэ.

In [ ]:
DB = {
    "plan": {
        "Даваа":  ["Squat 3×12", "Push-up 3×10", "Планк 3×30с"],
        "Лхагва": ["Jumping jack 3×30с", "Бёрпи 3×8", "Гүйлт 15 мин"],
        "Баасан": ["Lunge 3×10", "Гүүр 3×15", "Сунгалт 10 мин"],
    },
    "weights": [64.0, 63.2, 62.4],
    "memories": [],
}

def get_workout_plan(day: str) -> dict:
    """Тухайн гарагийн дасгалын жагсаалтыг буцаана.

    Args:
        day: Гарагийн нэр монголоор, жишээ нь "Даваа", "Лхагва", "Баасан".
    """
    return {"day": day, "exercises": DB["plan"].get(day, []) or "Амралтын өдөр"}

def update_workout_day(day: str, exercises: list[str]) -> dict:
    """Тухайн гарагийн дасгалыг шинэ жагсаалтаар СОЛИНО. Хэрэглэгч өөрчлөлт хүссэн үед л дуудна.

    Args:
        day: Гарагийн нэр монголоор.
        exercises: Шинэ дасгалууд, жишээ нь ["Хурдан алхалт 25 мин", "Гүүр 2×15"].
    """
    DB["plan"][day] = list(exercises)
    return {"ok": True, "day": day, "exercises": DB["plan"][day]}

def log_weight(kg: float) -> dict:
    """Хэрэглэгчийн өнөөдрийн жинг (кг) бүртгэнэ."""
    DB["weights"].append(float(kg))
    return {"ok": True, "kg": kg}

def get_progress() -> dict:
    """Эхлэлээс хойш жин хэдэн кг өөрчлөгдсөнийг буцаана."""
    w = DB["weights"]
    return {"start": w[0], "now": w[-1], "change_kg": round(w[-1] - w[0], 1)}

def save_memory(text: str) -> dict:
    """Хэрэглэгчийн тухай удаан хадгалах чухал мэдээллийг (гэмтэл, харшил, завтай цаг) санах ойд хадгална."""
    DB["memories"].append(text)
    return {"ok": True, "saved": text}

TOOLS = [get_workout_plan, update_workout_day, log_weight, get_progress, save_memory]
TOOL_MAP = {f.__name__: f for f in TOOLS}

print(get_workout_plan("Даваа"))   # хэрэгсэл бол жирийн функц — өөрөө дуудаж шалгаж болно

## Алхам 3. Модель өөрөө хэрэгсэл **сонгоно** (function calling)

Хэрэгслүүдийг моделд "цэс" болгож өгнө. Модель хариулахын оронд *"энэ хэрэгслийг ийм аргументтай дуудаач"* гэж хүсэлт буцаана.
`automatic_function_calling` унтраасан — давталтыг **бид өөрсдөө** бичих учраас (Алхам 4).

In [ ]:
SYSTEM = (
    "Чи FitCoach — монгол оюутны фитнесс дасгалжуулагч. Монголоор, богино хариул. "
    "Төлөвлөгөө, жин, явцын тухай асуувал таамаглахгүй, заавал хэрэгсэл дуудаж шалга. "
    "Өөрчлөлт хийсэн гэж зөвхөн хэрэгсэл амжилттай ажилласан үед л хэл. "
    "Гэмтэл, харшил гэх мэт удаан хадгалах мэдээллийг save_memory-оор хадгал."
)

def config(memories):
    mem = "\n".join("- " + m for m in memories) or "(хоосон)"
    return types.GenerateContentConfig(
        system_instruction=SYSTEM + "\n\nСанах ой:\n" + mem,
        tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

# --- Демо горимын "модель": түлхүүргүй үед энгийн дүрмээр шийднэ (FitCoach-ийн демо горимтой адил) ---
class _Call:
    def __init__(self, name, args): self.name, self.args = name, args
class _Cand:
    def __init__(self, content): self.content = content
class _Resp:
    def __init__(self, calls=None, text=None):
        self.function_calls = [_Call(n, a) for n, a in (calls or [])] or None
        self.text = text
        parts = ([types.Part.from_function_call(name=n, args=a) for n, a in calls] if calls
                 else [types.Part.from_text(text=text or "")])
        self.candidates = [_Cand(types.Content(role="model", parts=parts))]

def _demo_model(contents):
    last = contents[-1]
    if last.role == "tool":                       # хэрэгслийн үр дүн ирсэн → эцсийн хариу
        res = [p.function_response.response.get("result") for p in last.parts]
        return _Resp(text="(демо) Хэрэгслийн үр дүн: " + "; ".join(map(str, res)))
    q = last.parts[0].text.lower()
    import re
    kg = re.search(r"(\d+(?:\.\d+)?)\s*кг", q)
    if kg:
        return _Resp(calls=[("log_weight", {"kg": float(kg.group(1))}), ("get_progress", {})])
    if "өвд" in q:
        return _Resp(calls=[("save_memory", {"text": "Өвдөг өвддөг — үсрэлттэй дасгалаас зайлсхийх"}),
                            ("update_workout_day", {"day": "Лхагва", "exercises": ["Хурдан алхалт 25 мин", "Гүүр 2×15", "Сунгалт 10 мин"]})])
    if "явц" in q:
        return _Resp(calls=[("get_progress", {})])
    for d in DB["plan"]:
        if d.lower() in q:
            return _Resp(calls=[("get_workout_plan", {"day": d})])
    if DB["memories"] and ("мэдэх" in q or "санах" in q):
        return _Resp(text="(демо) Санах ойд: " + "; ".join(DB["memories"]))
    return _Resp(text="(демо) Сайн байна уу! Дасгал, жин, явцын талаар асуугаарай.")

def ask_model(contents):
    """Моделийг нэг удаа дуудна. Буцаах хариунд текст ЭСВЭЛ хэрэгсэл дуудах хүсэлт байна."""
    if client:
        return client.models.generate_content(model=MODEL, contents=contents, config=config(DB["memories"]))
    return _demo_model(contents)

def user_msg(text):
    return types.Content(role="user", parts=[types.Part.from_text(text=text)])

r = ask_model([user_msg("Миний даваа гарагийн дасгал юу вэ?")])
print("Модель хэрэгсэл хүссэн:", [(c.name, dict(c.args)) for c in (r.function_calls or [])])
# 👉 Модель хариулахын оронд get_workout_plan(day="Даваа")-г дуудахыг хүссэн. Гэхдээ одоохондоо хэн ч дуудаагүй!

## Алхам 4. **Agent loop** — агентын зүрх

1. Моделийг дуудна
2. Хэрэгсэл хүсвэл → **Python функцийг ажиллуулж**, үр дүнг моделд буцааж өгнө → 1-р алхам руу
3. Хэрэгсэл хүсэхгүй бол → эцсийн хариу

`MAX_TURNS` — хязгааргүй давталтаас хамгаална (FitCoach-д 6).
`steps` — хэрэглэгчид "агент юу хийсэн"-ийг харуулна (FitCoach-ийн чатын доорх 🔧 мөрүүд).

In [ ]:
MAX_TURNS = 6

def run_agent(text, history=None):
    contents = (history or []) + [user_msg(text)]
    steps = []
    for turn in range(MAX_TURNS):
        r = ask_model(contents)
        if not r.function_calls:                       # 3. хэрэгсэл хэрэггүй → дууслаа
            print("\n".join("   🔧 " + s for s in steps))
            print("🤖", r.text)
            contents.append(r.candidates[0].content)
            return contents
        contents.append(r.candidates[0].content)      # моделийн "дуудах хүсэлт"-ийг түүхэнд нэмнэ
        parts = []
        for call in r.function_calls:                  # 2. хэрэгсэл бүрийг жинхэнээр ажиллуулна
            args = dict(call.args or {})
            try:
                result = TOOL_MAP[call.name](**args)
            except Exception as e:                     # алдааг ч моделд хэлнэ — тэр өөрөө засаж магадгүй
                result = {"error": str(e)}
            steps.append(f"{call.name}({args}) → {result}")
            parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        contents.append(types.Content(role="tool", parts=parts))
    print("⚠️ MAX_TURNS хүрлээ")
    return contents

h = run_agent("Миний даваа гарагийн дасгал юу вэ?")

In [ ]:
# Олон алхамтай даалгавар: санах ойд хадгалах + төлөвлөгөө өөрчлөх
h = run_agent("Өвдөг өвдөөд байна. Лхагвын үсрэлттэй дасгалуудыг өвдөгт ээлтэйгээр солиоч.")
print("\nЛхагва одоо:", DB["plan"]["Лхагва"])   # 👉 өгөгдөл ҮНЭХЭЭР өөрчлөгдсөн — энэ бол агент

In [ ]:
h = run_agent("Өнөөдөр 61.8 кг боллоо. Явц хэр байна?")

## Алхам 5. Санах ой

`save_memory` хадгалсан зүйлс **system prompt**-д нэмэгддэг (`config()` функцийг харна уу).
Тиймээс **шинэ яриа** (history хоосон) эхлүүлсэн ч агент мэдсээр байна.

In [ ]:
print("Санах ой:", DB["memories"])
_ = run_agent("Миний тухай юу мэдэх вэ? Ямар дасгал хийхгүй байх ёстой вэ?")   # history өгөөгүй!

## FitCoach апп дээр яг хаана байна вэ?

| Энэ notebook | FitCoach (`github.com/Erdenetogs/fitness-coach-ai-agent-v2`) |
|---|---|
| `ask_model()` | `app/agent/llm.py` (Claude API) |
| `TOOLS`, `get_workout_plan`… | `app/agent/tools.py` — 8 хэрэгсэл, Neon Postgres-д бичнэ |
| `run_agent()` давталт, `MAX_TURNS` | `app/agent/coach.py` → `_chat_live` |
| `SYSTEM` | `coach.py`-ийн `SYSTEM` (+ аюулгүй байдал, үнэнч байх дүрэм) |
| `DB["memories"]` | `memories` хүснэгт, `save_memory` / `forget_memory` |
| `steps` | `agent_steps` — чатын доорх 🔧 мөрүүд |
| `_demo_model` | API түлхүүргүй үеийн демо горим |

## 🧩 Дасгал (өөрөө хийж үзэх)

1. **`forget_memory(text)`** хэрэгсэл нэмээд "Өвдөг эдгэрсэн, мартаарай" гэж туршина.
2. `MAX_TURNS = 1` болгоод Алхам 4-ийн өвдөгний даалгаврыг дахин ажиллуулна — юу болов, яагаад?
3. `SYSTEM`-д "Цээж өвдвөл дасгалаа зогсоож 119 руу залга гэж хэл" дүрэм нэмээд шалгана.
4. `update_workout_day`-ийн docstring-ийг устгаад модель хэрэгслийг зөв ашиглаж чадаж байгаа эсэхийг харьцуулна.

### Эх сурвалж
- [Google Gen AI SDK (python-genai)](https://googleapis.github.io/python-genai) — function calling, `AutomaticFunctionCallingConfig`
- [Gemini models](https://ai.google.dev/models) — загварын нэрс